# 02주 · 대푯값·산포·정의에 따른 차이

공학통계학 CE202

**목표**: 자료·확률·분포·추정·검정의 원리 이해와 통계적 해석 심화

학습 주제: 대푯값 / 산포

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE202-statistics-for-engineers/week02/CE202_W02.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE202-statistics-for-engineers/week02)

질문·가정 확인 → 손계산·예측 → 계산·시각화 → 한 조건 변경 → 통계적 해석·한계 설명

셀 실행: Shift+Enter. 아래 코드의 None은 직접 채울 항목. 기준 결과와 조건을 바꾼 결과를 비교

핵심 실습: 손계산·원리와 계산식의 연결·조건 변경·해석. AI 활용은 개념 학습 후 선택 활동

[공통 학습 계획](https://github.com/steelpl/lecture/blob/main/CE202-statistics-for-engineers/COURSE_PLAN.md) · [학기 운영안내](https://github.com/steelpl/lecture/blob/main/CURRENT_OFFERING.md)


## 0. 실행 환경

필요한 패키지의 최소 버전 확인. 기본 코드와 보조 함수는 이번 주 실습에 필요한 도구. 세부 구현 암기 불필요


In [ ]:
# 최초 실행 시 필요한 버전만 설치. Colab에서 위에서 아래로 실행
import importlib.metadata as metadata, subprocess, sys
from packaging.version import Version
requirements = {"numpy": "1.26", "pandas": "2.2", "scipy": "1.16", "matplotlib": "3.10", "openpyxl": "3.1"}
for package, minimum in requirements.items():
    try:
        outdated = Version(metadata.version(package)) < Version(minimum)
    except metadata.PackageNotFoundError:
        outdated = True
    if outdated:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"{package}>={minimum}"])
print("실행 환경 준비 완료. 이미 불러온 패키지를 업데이트한 경우 런타임 재시작 후 처음부터 실행")

In [ ]:
import math, tempfile, base64, zipfile, io
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
WORK = Path(tempfile.mkdtemp(prefix="ce202-"))
rng = np.random.default_rng(2027)

def vector(values, minimum=1):
    x = np.asarray(values, dtype=float)
    if x.ndim != 1 or len(x) < minimum or not np.isfinite(x).all():
        raise ValueError("유한한 1차원 수치 자료와 최소 표본수 확인 필요")
    return x

def probabilities(values):
    p = vector(values)
    if np.any(p < 0) or not np.isclose(p.sum(), 1):
        raise ValueError("확률의 비음수성·합계 1 확인 필요")
    return p

def mean_ci(values, confidence=0.95):
    x = vector(values, 2)
    if not 0 < confidence < 1:
        raise ValueError("신뢰수준은 0과 1 사이")
    se = x.std(ddof=1) / np.sqrt(len(x))
    q = stats.t.ppf((1 + confidence) / 2, df=len(x)-1)
    return {"n": len(x), "mean": x.mean(), "se": se,
            "df": len(x)-1, "low": x.mean()-q*se, "high": x.mean()+q*se}

def wilson_interval(successes, n, confidence=0.95):
    if not isinstance(n, (int, np.integer)) or n <= 0 or not 0 <= successes <= n:
        raise ValueError("0 ≤ 성공수 ≤ 양의 정수 표본수 확인 필요")
    if int(successes) != successes or not 0 < confidence < 1:
        raise ValueError("정수 성공수·신뢰수준 확인 필요")
    z = stats.norm.ppf((1+confidence)/2)
    phat = successes / n
    den = 1 + z*z/n
    center = (phat + z*z/(2*n)) / den
    half = z / den * np.sqrt(phat*(1-phat)/n + z*z/(4*n*n))
    return center-half, center+half

def anova_summary(groups):
    arrays = [vector(g, 2) for g in groups]
    if len(arrays) < 2:
        raise ValueError("두 집단 이상 필요")
    n = np.array([len(g) for g in arrays]); means = np.array([g.mean() for g in arrays])
    grand = np.concatenate(arrays).mean()
    ss_between = np.sum(n*(means-grand)**2)
    ss_within = sum(np.sum((g-g.mean())**2) for g in arrays)
    df_between = len(arrays)-1; df_within = n.sum()-len(arrays)
    if ss_within <= 0:
        raise ValueError("집단 내 변동이 0이면 일반적인 F 계산 불가")
    F = (ss_between/df_between) / (ss_within/df_within)
    return {"ss_between": ss_between, "ss_within": ss_within,
            "df_between": df_between, "df_within": df_within,
            "F": F, "p": stats.f.sf(F, df_between, df_within)}

def regression(x, y):
    x = vector(x, 3); y = vector(y, 3)
    if len(x) != len(y) or np.ptp(x) == 0:
        raise ValueError("같은 길이·서로 다른 x 값 확인 필요")
    fit = stats.linregress(x, y)
    fitted = fit.intercept + fit.slope*x
    residual = y-fitted
    sse = np.sum(residual**2); sst = np.sum((y-y.mean())**2)
    if sst <= 0:
        raise ValueError("y 변동이 0이면 일반적인 R² 해석 불가")
    return {"slope": fit.slope, "intercept": fit.intercept, "r2": 1-sse/sst,
            "sse": sse, "mse_residual": sse/(len(x)-2), "mse_prediction": sse/len(x),
            "p": fit.pvalue, "stderr": fit.stderr, "fitted": fitted, "residual": residual}


<a id="s01"></a>

## 도수표 평균과 이상치

**먼저 예측**: 800 포함·제외 시 평균과 중앙값 중 어느 쪽의 변화가 큰가?

계산 전에 분모·단위·조건을 한 줄로 기록


In [ ]:
values=np.array([5,15,25,35,45]); frequencies=np.array([5,8,3,2,2])
weighted_mean=np.dot(values,frequencies)/frequencies.sum()
assert np.isclose(weighted_mean,19)
production=np.array([230,250,265,285,290,800])
print("도수표 평균:",weighted_mean)
print(pd.DataFrame({"자료":["원본 6개","800 제외 사본"],"평균":[production.mean(),production[:-1].mean()],"중앙값":[np.median(production),np.median(production[:-1])]}))

**조건 변경**: 800을 다른 값으로 바꾼 사본에서 두 대푯값 비교

위 코드를 복사한 셀에서 한 조건씩 변경. 기준 셀은 보존

**선택 심화 · AI 검토**: 개별값과 도수를 구분하고 가중평균·단위·원본 보존 조건 명시

**결과 검토**: 이상치 제거는 계산 편의를 위한 자동 절차가 아님

예측·손계산·변경 결과·개념 설명·적용 한계 기록


In [ ]:
# 변경한 코드·확인할 식을 이 셀에 추가


<a id="s02"></a>

## 사분위수·분산의 정의

**먼저 예측**: 같은 자료의 Q1이 도구마다 달라도 반드시 계산 오류인가?

계산 전에 분모·단위·조건을 한 줄로 기록


In [ ]:
qdata=np.array([16,25,4,18,11,13,20,8,11,9])
# 원본 위치 규칙: n*p가 정수이면 인접 두 값 평균, 아니면 올림 위치
ordered=np.sort(qdata)
def course_quantile(x,p):
    x=np.sort(vector(x)); h=len(x)*p
    if p==0:return x[0]
    if p==1:return x[-1]
    if float(h).is_integer():return (x[int(h)-1]+x[int(h)])/2
    return x[math.ceil(h)-1]
print("원본 위치 규칙:",[course_quantile(qdata,p) for p in [0.25,0.5,0.75]])
print("Excel QUARTILE.INC/NumPy linear:",np.quantile(qdata,[0.25,0.5,0.75],method="linear"))
spread=np.array([10,11,12,13,12,14,13,11,13,12,12,11])
print("표본분산:",spread.var(ddof=1),"모분산식:",spread.var(ddof=0))
assert np.isclose(spread.var(ddof=1),spread.var(ddof=0)*12/11)

**조건 변경**: 분산의 ddof만 1→0으로 변경. 사분위수 계산 규칙은 별도 기록

위 코드를 복사한 셀에서 한 조건씩 변경. 기준 셀은 보존

**선택 심화 · AI 검토**: 표본/모집단 구분과 사분위수 알고리즘을 명시한 계산 요청

**결과 검토**: 사분위수 위치 규칙과 QUARTILE.INC의 차이를 오류로 혼동하지 않음

예측·손계산·변경 결과·개념 설명·적용 한계 기록


In [ ]:
# 변경한 코드·확인할 식을 이 셀에 추가


<a id="s03"></a>

## Excel 값 대조

**먼저 예측**: 빈 셀을 0으로 읽으면 n과 평균이 어떻게 달라지는가?

계산 전에 분모·단위·조건을 한 줄로 기록


In [ ]:
male=np.array([181.0, 170.0, 179.0, 183.0, 178.0, 171.0, 177.0, 174.0, 163.0, 167.0, 163.0, 173.0, 178.0, 170.0, 167.0, 177.0, 176.0, 180.0, 169.0, 178.0, 173.0, 173.0, 171.0, 171.0, 170.0, 174.0, 180.0]); female=np.array([161.0, 160.0, 158.0, 169.0, 162.0, 163.0, 158.0, 160.0, 160.0, 158.0, 160.0, 165.0, 170.0, 152.0, 160.0, 162.0, 167.0, 168.0, 166.0, 164.0, 158.0, 160.0, 160.0, 159.0])
print(pd.DataFrame({"집단":["남학생","여학생"],"n":[len(male),len(female)],"평균":[male.mean(),female.mean()],"표본분산":[male.var(ddof=1),female.var(ddof=1)]}))
assert len(male)==27 and len(female)==24
assert np.isclose(male.mean(),173.55555555555554)
assert np.isclose(female.var(ddof=1),17.53623188405798)

**조건 변경**: 여학생 열의 끝 빈 셀을 0으로 채운 잘못된 사본과 비교

위 코드를 복사한 셀에서 한 조건씩 변경. 기준 셀은 보존

**선택 심화 · AI 검토**: 원본 열·유효 표본수 27/24·빈 셀 처리·단위 확인을 먼저 요청

**결과 검토**: 원본 파일에 단위 표기가 없으므로 임의 단위 추가 금지

예측·손계산·변경 결과·개념 설명·적용 한계 기록


In [ ]:
# 변경한 코드·확인할 식을 이 셀에 추가


<a id="s04"></a>

## 왜도·첨도와 보정

**먼저 예측**: 첨도 기준값 3과 초과첨도 기준값 0은 같은 정의인가?

계산 전에 분모·단위·조건을 한 줄로 기록


In [ ]:
shape_data=np.array([1,3,2,0,1,1,2,4,3],dtype=float)
m2=np.mean((shape_data-shape_data.mean())**2); m4=np.mean((shape_data-shape_data.mean())**4)
course_beta=np.sum(((shape_data-shape_data.mean())/shape_data.std(ddof=1))**4)/(len(shape_data)-1)
print({"원본 식 beta":course_beta,"Pearson 적률 첨도":m4/m2**2,"보정된 초과첨도 Excel KURT":stats.kurtosis(shape_data,fisher=True,bias=False),"보정 왜도 Excel SKEW":stats.skew(shape_data,bias=False)})
plt.hist(shape_data,bins=np.arange(-0.5,5));plt.show()

**조건 변경**: 극단값을 하나 추가한 사본에서 왜도·첨도·그림을 함께 비교

위 코드를 복사한 셀에서 한 조건씩 변경. 기준 셀은 보존

**선택 심화 · AI 검토**: Pearson/초과첨도·bias 보정 여부를 명시하고 값 차이의 원인 설명 요청

**결과 검토**: 첨도만으로 봉우리 모양·정규성을 단정하지 않음

예측·손계산·변경 결과·개념 설명·적용 한계 기록


In [ ]:
# 변경한 코드·확인할 식을 이 셀에 추가


## 주차 확인

도수표 평균과 분산 분모, 분위수 정의의 관계 기록

입력 항목:

- `weighted_mean`: 도수표의 가중평균 · 수치
- `sample_variance_denominator`: 산포 예제 12개 값의 표본분산 분모 · 수치
- `different_quantile_rules_can_differ`: 분위수 알고리즘이 다르면 같은 자료에서 값이 달라도 되는가? · Python `True`/`False`

공개 확인용 항목. 결과값만 복사한 경우 학습 완료로 판단 불가. 분석 가정·자료 출처·추론 범위는 아래 기록과 함께 검토


In [ ]:
answers = {'weighted_mean': None, 'sample_variance_denominator': None, 'different_quantile_rules_can_differ': None}


In [ ]:
def check_answers(values):
    expected = {'weighted_mean': 19.0, 'sample_variance_denominator': 11, 'different_quantile_rules_can_differ': True}
    pending, wrong = [], []
    for key, target in expected.items():
        got = values.get(key)
        if got is None:
            pending.append(key)
        elif isinstance(target, bool):
            if not isinstance(got, (bool, np.bool_)) or bool(got) != target:
                wrong.append(key)
        elif isinstance(target, str):
            if not isinstance(got, str) or got.strip().lower() != target:
                wrong.append(key)
        elif isinstance(got, (bool, np.bool_)) or not isinstance(got, (int, float, np.number)) or not np.isfinite(got) or not np.isclose(got, target, rtol=1e-6, atol=1e-8):
            wrong.append(key)
    if pending:
        print("미완료:", ", ".join(pending))
    if wrong:
        print("재검토:", ", ".join(wrong))
    if not pending and not wrong:
        print("공개 계산·개념 확인 통과. 해석·가정·AI 검토 기록은 별도 확인 필요")
    return {"pending": pending, "wrong": wrong, "passed": not pending and not wrong}

check_result = check_answers(answers)

## 원리 탐구·결과 해석 기록

1. 문제·자료·단위·모형과 가정
2. 실행 전 예측·작은 손계산과 근거
3. 기준 계산의 중간값·결과와 원리의 연결
4. 한 조건 변경 결과와 차이의 원인
5. 독립 검산·적용 조건·불확실성·한계
6. 자신의 말로 설명할 핵심 원리 한 가지

AI 활용은 선택. 사용 시 요청·의심한 출력·수정·검산 근거를 별도 기록. 글자 수 자동 채점 없음


In [ ]:
review_log = {
    "문제·자료·단위·가정": None,
    "예측·손계산 근거": None,
    "중간 계산·원리 연결": None,
    "조건 변경·차이의 원인": None,
    "검산·해석·적용 한계": None,
    "내 설명": None,
}
# 선택 활동: AI 활용 시 요청·의심한 출력·수정·검산 근거를 별도 기록
